# Notebook 1 — Environment & Dataset Setup

Glaucoma Detection using ML/DL — reproduces the verified data pipeline: dataset
discovery, label identification, exact + near-duplicate checking, a seeded
stratified 70/15/15 split, and materialization of the final train/val/test
folders.

**Locked-in methodology decisions this notebook implements or respects:**
- Random seed = 42 everywhere a random choice is made.
- Split is stratified jointly by **source dataset × class**, not class alone,
  because per-source glaucoma prevalence varies substantially (10%–29%).
- Counts are *discovered from the raw files on disk*, not hard-coded. A
  previously-recorded reference is used only as a regression sanity check
  (see the verification cell below), never as an input to computation.
- GPU/PyTorch setup is intentionally **not** part of this notebook — no
  training happens here. That belongs to Notebook 3.


## 1. Environment setup and package installation

In [ ]:
import importlib, subprocess, sys

REQUIRED_PACKAGES = ["pandas", "numpy", "Pillow"]

def _ensure(pkg):
    import_name = "PIL" if pkg == "Pillow" else pkg
    try:
        importlib.import_module(import_name)
        print(f"[ok]      {pkg} already available")
    except ImportError:
        print(f"[install] {pkg} not found, attempting pip install ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            print(f"[ok]      {pkg} installed")
        except Exception as e:
            print(f"[warning] could not install {pkg} automatically ({e}). "
                  f"Install it manually if a later cell fails on import.")

for _pkg in REQUIRED_PACKAGES:
    _ensure(_pkg)

# Standard library
import os, sys, json, csv, hashlib, shutil, random, time
from pathlib import Path
from collections import Counter, defaultdict

# Third-party
import pandas as pd
import numpy as np
from PIL import Image

print()
print("Environment ready.")
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. Configuration and reproducibility settings

Every random operation in this notebook (shuffling before the stratified
split) is seeded. There is no model training in this notebook, so no
PyTorch/CUDA determinism settings are needed yet — those belong in
Notebook 3.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed set to {SEED} (random, numpy).")

## 3. Dataset path configuration

**This is the single cell to edit when moving from the local Mac to Google
Drive/Colab.** Change `RUNTIME_ENV` to `"colab_drive"` and fill in the paths
in `PATH_PRESETS["colab_drive"]` — nothing else in this notebook needs to
change, because every later cell reads `RAW_DATASET_ROOT` / `OUTPUT_ROOT`
rather than hard-coding a path.

In [ ]:
RUNTIME_ENV = "local_mac"   # "local_mac" or "colab_drive"

PATH_PRESETS = {
    "local_mac": {
        # Raw, extracted Kaggle download (contains ORIGA/, G1020/, REFUGE/, models/)
        "RAW_DATASET_ROOT": os.path.expanduser("~/Downloads/archive"),
        # Where this project's outputs (split dataset, manifests, results) live
        "OUTPUT_ROOT": os.path.expanduser("~/Downloads/GlaucomaProject"),
    },
    "colab_drive": {
        # Fill these in once Google Drive is mounted in Colab (Notebook 1's
        # Drive-migration step). Left as placeholders on purpose.
        "RAW_DATASET_ROOT": "/content/drive/MyDrive/GlaucomaProject/raw_kaggle_download",
        "OUTPUT_ROOT": "/content/drive/MyDrive/GlaucomaProject",
    },
}

_paths = PATH_PRESETS[RUNTIME_ENV]
RAW_DATASET_ROOT = _paths["RAW_DATASET_ROOT"]
OUTPUT_ROOT = _paths["OUTPUT_ROOT"]

DATASET_OUT_DIR = os.path.join(OUTPUT_ROOT, "dataset")
RESULTS_OUT_DIR = os.path.join(OUTPUT_ROOT, "results")
for d in [DATASET_OUT_DIR, RESULTS_OUT_DIR]:
    for split in ["train", "val", "test"]:
        for cls in ["glaucoma", "normal"]:
            os.makedirs(os.path.join(DATASET_OUT_DIR, split, cls), exist_ok=True)
    os.makedirs(RESULTS_OUT_DIR, exist_ok=True)

assert os.path.isdir(RAW_DATASET_ROOT), (
    f"RAW_DATASET_ROOT does not exist: {RAW_DATASET_ROOT}. "
    f"Update PATH_PRESETS['{RUNTIME_ENV}'] above."
)

print("RUNTIME_ENV     :", RUNTIME_ENV)
print("RAW_DATASET_ROOT:", RAW_DATASET_ROOT, "(exists)")
print("OUTPUT_ROOT     :", OUTPUT_ROOT)
print("DATASET_OUT_DIR :", DATASET_OUT_DIR)
print("RESULTS_OUT_DIR :", RESULTS_OUT_DIR)

## 4. Discovery/inspection of the raw Kaggle dataset

Lists what is actually present under `RAW_DATASET_ROOT` rather than assuming
a folder layout. Nothing downstream is hard-coded from this point without
first checking it exists.

In [ ]:
def list_tree(root, max_depth=2, _depth=0, _prefix=""):
    if _depth > max_depth:
        return
    try:
        entries = sorted(os.listdir(root))
    except NotADirectoryError:
        return
    for name in entries:
        p = os.path.join(root, name)
        tag = "/" if os.path.isdir(p) else ""
        print(f"{_prefix}{name}{tag}")
        if os.path.isdir(p):
            list_tree(p, max_depth=max_depth, _depth=_depth + 1, _prefix=_prefix + "    ")

top_level = sorted(os.listdir(RAW_DATASET_ROOT))
print("Top-level entries under RAW_DATASET_ROOT:")
for e in top_level:
    kind = "dir" if os.path.isdir(os.path.join(RAW_DATASET_ROOT, e)) else "file"
    print(f"  [{kind}] {e}")

print()
print("Full tree (depth 2):")
list_tree(RAW_DATASET_ROOT, max_depth=2)

## 5. Identification of ORIGA, G1020, and REFUGE source datasets

Each candidate source is only accepted if its *marker file* (something that
uniquely identifies that dataset's own label/index file) is actually found on
disk — this is a check, not an assumption. REFUGE's train/val/test are
discovered as independent splits, since (as this notebook will show in the
next section) they don't all carry usable labels.

In [ ]:
# Known signature files for each candidate source. A source is only used if
# its marker is actually found under RAW_DATASET_ROOT.
KNOWN_SOURCE_MARKERS = {
    "ORIGA":        {"marker": "ORIGA/OrigaList.csv",        "images_subdir": "ORIGA/Images"},
    "G1020":        {"marker": "G1020/G1020.csv",            "images_subdir": "G1020/Images"},
    "REFUGE_train": {"marker": "REFUGE/train/index.json",    "images_subdir": "REFUGE/train/Images"},
    "REFUGE_val":   {"marker": "REFUGE/val/index.json",      "images_subdir": "REFUGE/val/Images"},
    "REFUGE_test":  {"marker": "REFUGE/test/index.json",     "images_subdir": "REFUGE/test/Images"},
}

discovered_sources = {}
print("Source discovery:")
for name, spec in KNOWN_SOURCE_MARKERS.items():
    marker_path = os.path.join(RAW_DATASET_ROOT, spec["marker"])
    images_path = os.path.join(RAW_DATASET_ROOT, spec["images_subdir"])
    found = os.path.isfile(marker_path)
    images_found = os.path.isdir(images_path)
    status = "FOUND" if (found and images_found) else "NOT FOUND"
    print(f"  {name:14s} marker={spec['marker']:<28s} -> {status}")
    if found and images_found:
        discovered_sources[name] = spec

print()
print(f"Accepted sources ({len(discovered_sources)}):", list(discovered_sources.keys()))
assert len(discovered_sources) > 0, "No known source datasets were found under RAW_DATASET_ROOT."

## 6. Identification of glaucoma vs normal labels

Each source stores labels in a different format. Label convention used
throughout this project: **0 = Normal, 1 = Glaucoma**. Rather than assuming
every discovered source has usable labels, this cell actively checks for a
`Label`-bearing field and reports sources/splits that don't have one — this
is how the notebook *discovers* that REFUGE's official test split ships
without public ground truth, instead of being told.

In [ ]:
def load_origa_labels(root):
    path = os.path.join(root, "ORIGA", "OrigaList.csv")
    labels = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            labels[row["Filename"]] = int(row["Glaucoma"])
    return labels

def load_g1020_labels(root):
    path = os.path.join(root, "G1020", "G1020.csv")
    labels = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            labels[row["imageID"]] = int(row["binaryLabels"])
    return labels

def load_refuge_labels(root, split):
    """Returns (labels_dict, has_labels: bool). REFUGE's official test split
    is expected to come back with has_labels=False -- verified, not assumed."""
    path = os.path.join(root, "REFUGE", split, "index.json")
    with open(path) as f:
        idx = json.load(f)
    sample = next(iter(idx.values())) if idx else {}
    has_labels = "Label" in sample
    labels = {v["ImgName"]: v["Label"] for v in idx.values()} if has_labels else {}
    return labels, has_labels

label_loaders = {}
usable_sources = {}

print("Label discovery per source:")
for name in discovered_sources:
    if name == "ORIGA":
        labels = load_origa_labels(RAW_DATASET_ROOT)
        has_labels = len(labels) > 0
    elif name == "G1020":
        labels = load_g1020_labels(RAW_DATASET_ROOT)
        has_labels = len(labels) > 0
    elif name.startswith("REFUGE_"):
        split = name.split("_", 1)[1]
        labels, has_labels = load_refuge_labels(RAW_DATASET_ROOT, split)
    else:
        labels, has_labels = {}, False

    label_loaders[name] = labels
    status = f"labeled ({len(labels)} entries)" if has_labels else "NO LABELS -- excluded from classification pool"
    print(f"  {name:14s} -> {status}")
    if has_labels:
        usable_sources[name] = discovered_sources[name]

print()
print(f"Sources usable for classification ({len(usable_sources)}):", list(usable_sources.keys()))
excluded = set(discovered_sources) - set(usable_sources)
if excluded:
    print(f"Excluded (no labels found): {sorted(excluded)}")

## 7. Exclusion of segmentation masks and other non-classification files

Every source folder contains far more than classification images: OD/OC
segmentation masks, resized/cropped duplicates of the same photos
(`*_Square`, `*_Cropped`), and in one case pretrained model weights. This
cell explicitly enumerates every subfolder under each usable source and
classifies it as *images* (the one canonical folder actually used) or
*excluded*, with a reason -- so nothing is silently mixed in.

In [ ]:
def classify_subdir(name):
    lname = name.lower()
    if "mask" in lname:
        return "excluded", "segmentation mask, not a classification image"
    if "square" in lname or "cropped" in lname:
        return "excluded", "resized/cropped derivative of the same photos (would duplicate the canonical Images folder)"
    if "nerveremoved" in lname:
        return "excluded", "processed derivative image, not the canonical photo"
    if "annotation" in lname:
        return "excluded", "annotation metadata, not an image folder"
    if lname in ("gts", "illustrations"):
        return "excluded", "ground-truth mask / illustration folder, not classification images"
    if lname == "images":
        return "images", "canonical classification image folder"
    return "excluded", "unrecognized folder, conservatively excluded"

exclusion_log = []
IMAGES_SUBDIR = {}

for name, spec in usable_sources.items():
    # top folder for this source, e.g. "ORIGA" or "REFUGE/train"
    top_rel = os.path.dirname(spec["images_subdir"]) if "/" in spec["images_subdir"] else spec["images_subdir"].rsplit("/", 1)[0]
    top_abs = os.path.join(RAW_DATASET_ROOT, top_rel)
    subdirs = sorted(d for d in os.listdir(top_abs) if os.path.isdir(os.path.join(top_abs, d)))
    print(f"{name}  (under {top_rel}/):")
    chosen = None
    for sd in subdirs:
        kind, reason = classify_subdir(sd)
        print(f"    [{kind:8s}] {sd:28s} -- {reason}")
        exclusion_log.append({"source": name, "subdir": f"{top_rel}/{sd}", "classification": kind, "reason": reason})
        if kind == "images" and os.path.join(top_rel, sd) == spec["images_subdir"]:
            chosen = sd
    IMAGES_SUBDIR[name] = spec["images_subdir"]
    print(f"    -> using: {spec['images_subdir']}")
    print()

exclusion_df = pd.DataFrame(exclusion_log)
print(f"Total subfolders classified: {len(exclusion_df)} "
      f"({(exclusion_df['classification']=='images').sum()} used as images, "
      f"{(exclusion_df['classification']=='excluded').sum()} excluded)")

## 8. Exact image counts

Builds the raw manifest by scanning each source's canonical image folder on
disk and joining against the labels discovered in section 6 -- the counts
below are computed, not stated.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

raw_rows = []
unmatched = []
for name, images_subdir in IMAGES_SUBDIR.items():
    folder = os.path.join(RAW_DATASET_ROOT, images_subdir)
    files = sorted(f for f in os.listdir(folder) if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS)
    labels = label_loaders[name]
    for fn in files:
        if fn in labels:
            raw_rows.append({"source": name, "filename": fn, "label": labels[fn]})
        else:
            unmatched.append((name, fn))

raw_df = pd.DataFrame(raw_rows)
print(f"Discovered image files with a matching label: {len(raw_df)}")
if unmatched:
    print(f"WARNING: {len(unmatched)} image files had no matching label entry and were skipped, e.g. {unmatched[:5]}")
else:
    print("Every discovered image file had a matching label entry.")

print()
print("=== Class counts (computed) ===")
class_counts = raw_df["label"].value_counts().rename({0: "Normal", 1: "Glaucoma"})
print(class_counts)

TOTAL = len(raw_df)
NORMAL = int((raw_df["label"] == 0).sum())
GLAUCOMA = int((raw_df["label"] == 1).sum())
print()
print(f"TOTAL={TOTAL}  NORMAL={NORMAL}  GLAUCOMA={GLAUCOMA}")

## 9. Exact-vs-expected dataset verification

`EXPECTED_REFERENCE` below is the result of a prior *manual* verification
pass (recorded 2026-08-15) -- it is used **only** as a regression sanity
check printed after the fact. Every cell above and below computes its own
numbers independently; nothing upstream reads from this dict.

In [ ]:
EXPECTED_REFERENCE = {"total": 2470, "normal": 1926, "glaucoma": 544}  # prior manual pass, sanity check only

def check(label, actual, expected):
    status = "PASS" if actual == expected else "MISMATCH"
    print(f"  [{status}] {label}: computed={actual}  expected(reference)={expected}")
    return status == "PASS"

print("Regression check against prior manually-verified reference:")
all_pass = True
all_pass &= check("total",    TOTAL,    EXPECTED_REFERENCE["total"])
all_pass &= check("normal",   NORMAL,   EXPECTED_REFERENCE["normal"])
all_pass &= check("glaucoma", GLAUCOMA, EXPECTED_REFERENCE["glaucoma"])
print()
print("Overall:", "MATCHES prior verified reference" if all_pass else
                   "DOES NOT MATCH reference -- investigate before proceeding (dataset may have changed).")

## 10. Exact duplicate detection (MD5)

Hashes the raw bytes of every discovered image file. Any two files with an
identical MD5 are byte-for-byte the same file.

In [ ]:
def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

t0 = time.time()
md5_list = []
for row in raw_df.itertuples(index=False):
    path = os.path.join(RAW_DATASET_ROOT, IMAGES_SUBDIR[row.source], row.filename)
    md5_list.append(file_md5(path))
raw_df = raw_df.copy()
raw_df["md5"] = md5_list
print(f"Hashed {len(raw_df)} files in {time.time()-t0:.1f}s")

dup_mask = raw_df.duplicated(subset="md5", keep=False)
exact_dups = raw_df[dup_mask]
print(f"Files involved in an exact (MD5) duplicate: {len(exact_dups)}")
if len(exact_dups):
    for md5_val, g in exact_dups.groupby("md5"):
        print("  ", list(zip(g["source"], g["filename"], g["label"])))
else:
    print("No exact duplicates found.")

## 11. Near-duplicate check (documented, computationally reasonable)

**Note on methodology, including a correction made during this project:**
an early pass used a loose dHash Hamming-distance threshold (<=6 out of 64
bits) and flagged ~129,000 "candidate" pairs -- almost all false positives,
because fundus photographs are visually homogeneous (a bright circular disc
on a similar-colored background), so a coarse global hash alone is not
discriminative enough in this domain. A tighter threshold alone did not fix
this either: taking *every* pair within distance <=3 still produced ~23,700
candidates, because many images have several similarly-close neighbors, not
just one -- checking all of them at the pixel level would be far too slow to
be "computationally reasonable."

The fix used here: for **each image, keep only its single closest
neighbor** (nearest-neighbor search, not all-pairs-within-threshold). That
bounds the candidate set to at most `n` pairs (2,470 here) regardless of how
many images happen to look similar, and is the right question anyway -- "is
this image a near-duplicate of *anything* else" only needs the closest
match. A pair is only reported as a **confirmed** near-duplicate if **two
independent signals agree**: dHash Hamming distance <= `HASH_DIST_THRESHOLD`
*and* pixel-level mean absolute difference (MAE) on a 128x128 resize <=
`PIXEL_MAE_THRESHOLD`.

Cost: the nearest-neighbor search is O(n^2) hash comparisons
(~2,470^2/2 ~= 3M operations, vectorized with NumPy popcount, a few seconds),
and pixel-level MAE is computed for at most `n` pairs. Both are comfortably
reasonable at this dataset size; a much larger dataset would need an
approximate nearest-neighbor index instead of the full O(n^2) pass.

**On the MAE threshold itself:** the pixel-MAE values among hash-candidates
form a continuous distribution with no sharp gap separating "true
duplicates" from "coincidentally similar" -- expected, given how visually
homogeneous fundus photos are. There is no fully principled cutoff. This
notebook uses `PIXEL_MAE_THRESHOLD = 5.0` as a deliberately conservative
choice (it isolates a small, distinctly tighter cluster at the low end of
the observed distribution) and prints every confirmed pair for **manual
visual review** -- it does not silently drop or merge anything on your
behalf. Pay particular attention to any confirmed pair with *different*
labels: that would mean the same or near-identical photo is recorded as
both glaucoma and normal, which is a data-quality issue worth investigating
directly, not just a leakage concern.

In [ ]:
HASH_DIST_THRESHOLD = 3     # out of 64 bits -- tight, candidates only
PIXEL_MAE_THRESHOLD = 5.0   # out of 255 -- see note above on why this is conservative, not a hard cutoff

def dhash(img, hash_size=8):
    """Difference hash: resize to (hash_size+1, hash_size) grayscale, compare
    adjacent pixel brightness. Returns a hash_size*hash_size-bit integer."""
    g = img.convert("L").resize((hash_size + 1, hash_size), Image.LANCZOS)
    arr = np.asarray(g, dtype=np.int16)
    bits = (arr[:, 1:] > arr[:, :-1]).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h

def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0f0f0f0f0f0f0f0f)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

t0 = time.time()
dhash_list = []
for row in raw_df.itertuples(index=False):
    path = os.path.join(RAW_DATASET_ROOT, IMAGES_SUBDIR[row.source], row.filename)
    with Image.open(path) as img:
        img.load()
        dhash_list.append(dhash(img))
raw_df["dhash"] = dhash_list
print(f"Computed dHash for {len(raw_df)} images in {time.time()-t0:.1f}s")

In [ ]:
hashes = raw_df["dhash"].to_numpy(dtype=np.uint64)
n = len(hashes)

# Nearest-neighbor search: for each image, find its single closest match by
# Hamming distance (bounds the candidate set to <= n pairs, see note above).
t0 = time.time()
min_dist = np.full(n, 65, dtype=np.int64)
nn_idx = np.full(n, -1, dtype=np.int64)
for i in range(n):
    xor = np.bitwise_xor(hashes, hashes[i]).astype(np.uint64)
    dist = popcount64(xor).astype(np.int64)
    dist[i] = 999  # exclude self
    j = int(np.argmin(dist))
    min_dist[i] = dist[j]
    nn_idx[i] = j
print(f"Nearest-neighbor hash search over {n} images done in {time.time()-t0:.1f}s")

candidate_pairs = []
seen_pairs = set()
for i in range(n):
    if min_dist[i] <= HASH_DIST_THRESHOLD:
        j = int(nn_idx[i])
        key = tuple(sorted((i, j)))
        if key not in seen_pairs:
            seen_pairs.add(key)
            candidate_pairs.append((key[0], key[1], int(min_dist[i])))

print(f"Nearest-neighbor candidates with distance <= {HASH_DIST_THRESHOLD}/64: {len(candidate_pairs)}")

def pixel_mae(path_a, path_b, size=128):
    a = np.asarray(Image.open(path_a).convert("RGB").resize((size, size)), dtype=np.float32)
    b = np.asarray(Image.open(path_b).convert("RGB").resize((size, size)), dtype=np.float32)
    return float(np.abs(a - b).mean())

confirmed = []
for i, j, dist in candidate_pairs:
    ri, rj = raw_df.iloc[i], raw_df.iloc[j]
    pi = os.path.join(RAW_DATASET_ROOT, IMAGES_SUBDIR[ri.source], ri.filename)
    pj = os.path.join(RAW_DATASET_ROOT, IMAGES_SUBDIR[rj.source], rj.filename)
    mae = pixel_mae(pi, pj)
    if mae <= PIXEL_MAE_THRESHOLD:
        confirmed.append({"source_a": ri.source, "file_a": ri.filename, "label_a": ri.label,
                           "source_b": rj.source, "file_b": rj.filename, "label_b": rj.label,
                           "hash_dist": dist, "pixel_mae": round(mae, 2)})

near_dup_df = pd.DataFrame(confirmed)
print(f"Confirmed near-duplicate candidates (hash AND pixel-MAE agree, threshold={PIXEL_MAE_THRESHOLD}): {len(near_dup_df)}")
if len(near_dup_df):
    print(near_dup_df.to_string(index=False))
    label_conflicts = near_dup_df[near_dup_df["label_a"] != near_dup_df["label_b"]]
    if len(label_conflicts):
        print()
        print(f"*** WARNING: {len(label_conflicts)} confirmed pair(s) have DIFFERENT labels on "
              f"near-identical images -- inspect these directly, this is a data-quality issue, "
              f"not just a leakage risk: ***")
        print(label_conflicts.to_string(index=False))
    print()
    print("These are candidates for manual visual review, not automatically removed. "
          "Decide case-by-case whether to drop one image from a pair before training.")
else:
    print("None of the hash candidates were confirmed as real near-duplicates at the pixel level "
          "-- they were visually similar-but-distinct fundus photos.")

## 12. Master manifest

Combines everything discovered so far into one manifest with, at minimum,
`filepath`, `filename`, `source`, and `label` columns. `filepath` is
resolved from the *configured* `RAW_DATASET_ROOT` + `IMAGES_SUBDIR`, so it
stays correct if section 3's config is switched to Colab/Drive.

In [ ]:
def resolve_filepath(source, filename):
    return os.path.join(RAW_DATASET_ROOT, IMAGES_SUBDIR[source], filename)

manifest_df = raw_df.copy()
manifest_df["filepath"] = [resolve_filepath(s, f) for s, f in zip(manifest_df["source"], manifest_df["filename"])]
manifest_df["class_name"] = manifest_df["label"].map({0: "normal", 1: "glaucoma"})

manifest_df = manifest_df[["filepath", "filename", "source", "label", "class_name", "md5", "dhash"]]
print(f"Master manifest columns: {list(manifest_df.columns)}")
print(f"Master manifest rows: {len(manifest_df)}")
manifest_df.head()

## 13. Resolve near-duplicate data-quality issues (label-conflict exclusion + same-eye grouping)

**Decision (made by the user after manually reviewing the confirmed
near-duplicate pairs from section 11 -- see `claude/notebook1_log.md` in the
project for the full visual/pixel investigation this decision is based on):**

1. The one pair with **conflicting labels** -- `G1020/image_1982.jpg`
   (Normal) and `G1020/image_2658.jpg` (Glaucoma) -- is almost certainly the
   same eye photographed twice (near-identical peripheral vessel pattern;
   real pixel differences concentrated at the optic disc, zoomed-region MAE
   10.18 vs. 3.39 full-image). Since a single eye's glaucoma status cannot
   genuinely differ between two closely-timed exams, one of the two labels is
   very likely a data-curation error in the underlying G1020 release -- but
   *which* label is correct cannot be determined from the image or the
   available metadata (`G1020.csv` has no patient/session ID, and both
   images' LabelMe JSON annotations are independently drawn). Rather than
   guess, **both images are excluded from the classification pool**, with the
   reason recorded explicitly in the master manifest as
   `excluded_conflicting_near_duplicate_label`.
2. The remaining near-duplicate pairs (labels agree on both sides) are kept,
   but since they are still very likely the same eye photographed twice, they
   must never be split across train/val/test -- that would leak the same
   eye's appearance between training and evaluation. This is implemented
   generally below: every pair of label-agreeing near-duplicate images is
   unioned into a **group**, and the split (section 14) treats each group as
   a single, indivisible unit -- so pairs that already happen to sit in the
   same split (e.g. two pairs that landed entirely in train) are simply kept
   together, while pairs that currently cross a split boundary get moved as
   a unit the next time the split is (re)computed.

Both steps below operate on `near_dup_df`, the near-duplicate table computed
programmatically in section 11 -- not a hand-typed list of filenames -- so
re-running this notebook on a re-downloaded or updated dataset re-derives the
exclusions/groups from whatever near-duplicates are actually found, instead
of silently reusing a stale, one-time manual fix.

In [ ]:
# --- Step 1: exclude images involved in a label-conflicting near-duplicate pair ---
label_conflicts = near_dup_df[near_dup_df["label_a"] != near_dup_df["label_b"]]

excluded_keys = set()
exclusion_reason = {}
for r in label_conflicts.itertuples(index=False):
    for src, fn in [(r.source_a, r.file_a), (r.source_b, r.file_b)]:
        excluded_keys.add((src, fn))
        exclusion_reason[(src, fn)] = "excluded_conflicting_near_duplicate_label"

print(f"Label-conflicting near-duplicate pairs found: {len(label_conflicts)}")
print(f"Images excluded as a result: {len(excluded_keys)}")
for (src, fn) in sorted(excluded_keys):
    print(f"  EXCLUDED: {src}/{fn}  reason={exclusion_reason[(src, fn)]}")

manifest_df["excluded"] = [
    (s, f) in excluded_keys for s, f in zip(manifest_df["source"], manifest_df["filename"])
]
manifest_df["exclusion_reason"] = [
    exclusion_reason.get((s, f), "") for s, f in zip(manifest_df["source"], manifest_df["filename"])
]

n_before = len(manifest_df)
n_excluded = int(manifest_df["excluded"].sum())
print()
print(f"Master manifest rows: {n_before} total, {n_excluded} excluded, "
      f"{n_before - n_excluded} usable for the classification split.")

# --- Step 2: union-find same-eye groups from the remaining (label-agreeing) pairs ---
consistent_pairs = near_dup_df[near_dup_df["label_a"] == near_dup_df["label_b"]]
# defensive: drop any consistent-label pair that (unexpectedly) touches an excluded image
consistent_pairs = consistent_pairs[
    ~consistent_pairs.apply(
        lambda r: (r["source_a"], r["file_a"]) in excluded_keys or (r["source_b"], r["file_b"]) in excluded_keys,
        axis=1,
    )
]

parent = {}
def _find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        parent[rb] = ra

for s, f in zip(manifest_df["source"], manifest_df["filename"]):
    parent.setdefault((s, f), (s, f))

for r in consistent_pairs.itertuples(index=False):
    _union((r.source_a, r.file_a), (r.source_b, r.file_b))

manifest_df["group_id"] = [
    _find((s, f)) for s, f in zip(manifest_df["source"], manifest_df["filename"])
]

group_sizes = manifest_df[~manifest_df["excluded"]].groupby("group_id").size()
multi_groups = group_sizes[group_sizes > 1]
print()
print(f"Same-eye groups formed from {len(consistent_pairs)} label-agreeing near-duplicate pair(s): "
      f"{len(multi_groups)} group(s) of size > 1.")
for gid, size in multi_groups.items():
    members = manifest_df[manifest_df["group_id"] == gid][["source", "filename", "label"]]
    print(f"  group {gid}: {size} images")
    print(members.to_string(index=False))

## 14. Group-aware stratified 70/15/15 split by source x class (seed 42)

Same idea as a plain stratified split -- stratified jointly on `source x
label` so that both overall class balance and each source's contribution are
preserved -- but now the unit being split is a **group** from section 13
(almost always a single image; 6 groups have 2 images), not an individual
image, so that no same-eye/near-duplicate group is ever divided across
train/val/test. The two excluded images from section 13 are not assigned to
any split (`split = "excluded"`).

Within each `(source, label)` stratum, groups are shuffled deterministically
(seed 42) and then assigned greedily: each group in turn goes to whichever
split currently has the largest remaining deficit against its target image
count (`stratum_size * fraction`). This keeps every group intact while
tracking the target 70/15/15 proportions as closely as an integer,
group-respecting assignment allows -- with so few multi-image groups (6 out
of the usable pool), the resulting proportions are very close to a plain
70/15/15 split.

In [ ]:
def group_aware_stratified_split(df, frac_train=0.70, frac_val=0.15, frac_test=0.15, seed=SEED):
    assert abs(frac_train + frac_val + frac_test - 1.0) < 1e-9
    usable = df[~df["excluded"]].copy()

    groups = usable.groupby("group_id").agg(
        source=("source", "first"),
        label=("label", "first"),
        size=("filename", "count"),
    ).reset_index()

    # sanity check: every group must be internally consistent on source & label
    check_src = usable.groupby("group_id")["source"].nunique()
    check_lbl = usable.groupby("group_id")["label"].nunique()
    assert (check_src == 1).all(), "A group spans more than one source -- investigate."
    assert (check_lbl == 1).all(), "A group spans more than one label -- investigate."

    groups["stratum"] = groups["source"] + "_" + groups["label"].astype(str)

    rng = np.random.RandomState(seed)
    split_of_group = {}
    for stratum, g in groups.groupby("stratum"):
        stratum_total = int(g["size"].sum())
        target = {
            "train": stratum_total * frac_train,
            "val": stratum_total * frac_val,
            "test": stratum_total * frac_test,
        }
        assigned = {"train": 0, "val": 0, "test": 0}
        order = rng.permutation(len(g))
        g_shuffled = g.iloc[order]
        for row in g_shuffled.itertuples(index=False):
            deficits = {s: target[s] - assigned[s] for s in ("train", "val", "test")}
            best = max(deficits, key=deficits.get)
            split_of_group[row.group_id] = best
            assigned[best] += row.size

    usable["split"] = usable["group_id"].map(split_of_group)

    excluded_rows = df[df["excluded"]].copy()
    excluded_rows["split"] = "excluded"
    out = pd.concat([usable, excluded_rows], ignore_index=True)
    return out

manifest_df = group_aware_stratified_split(manifest_df, seed=SEED)
print("Group-aware split assigned. Seed:", SEED)
print(manifest_df.loc[manifest_df["split"] != "excluded", "split"].value_counts())
print()
print("Excluded (not part of any split):", int((manifest_df["split"] == "excluded").sum()))

## 15. Verification of the resulting distribution

Checked five ways: overall split sizes/proportions, class balance within
each split, source composition within each split, the full source x class x
split breakdown, and -- new -- confirmation that **no same-eye/near-duplicate
group is split across train/val/test**.

In [ ]:
usable_df = manifest_df[manifest_df["split"] != "excluded"]

print("=== Overall split sizes ===")
print(usable_df["split"].value_counts())
print()

print("=== Overall split proportions ===")
print((usable_df["split"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")
print()

print("=== Class distribution per split ===")
print(usable_df.groupby("split")["label"].value_counts().unstack(fill_value=0).rename(columns={0: "Normal", 1: "Glaucoma"}))
print()

print("=== Source distribution per split ===")
print(usable_df.groupby(["split", "source"]).size().unstack(fill_value=0))
print()

print("=== Source x class x split (full breakdown) ===")
print(usable_df.groupby(["source", "label", "split"]).size().unstack(fill_value=0))
print()

print("=== Same-eye/near-duplicate group cross-split check ===")
group_split_counts = usable_df.groupby("group_id")["split"].nunique()
crossing_groups = group_split_counts[group_split_counts > 1]
if len(crossing_groups) == 0:
    print(f"PASS: all {usable_df['group_id'].nunique()} groups are fully contained within a single split.")
else:
    print(f"FAIL: {len(crossing_groups)} group(s) span more than one split:")
    print(usable_df[usable_df["group_id"].isin(crossing_groups.index)]
          .sort_values("group_id")[["group_id", "source", "filename", "label", "split"]].to_string(index=False))

## 16. Materialization of train/val/test folders

Wipes and rebuilds `DATASET_OUT_DIR/{split}/{class}/` from scratch before
copying, rather than editing the previously materialized folders in place --
this guarantees the on-disk folders always exactly match the current
`manifest_df`, even when images move between splits or get excluded (as
happened here after section 13/14). Only rows with `split != "excluded"` are
copied; files are prefixed with their source name to avoid filename
collisions (e.g. `ORIGA/001.jpg` and `G1020/001.jpg` would otherwise
collide).

In [ ]:
t0 = time.time()

# Wipe existing materialized split folders so stale files (from a prior split
# assignment) cannot linger after re-running this notebook with a different split.
for split in ["train", "val", "test"]:
    for cls in ["glaucoma", "normal"]:
        d = os.path.join(DATASET_OUT_DIR, split, cls)
        if os.path.isdir(d):
            for fn in os.listdir(d):
                fp = os.path.join(d, fn)
                if os.path.isfile(fp):
                    os.remove(fp)
        os.makedirs(d, exist_ok=True)
print("Cleared existing materialized train/val/test folders.")

to_copy = manifest_df[manifest_df["split"] != "excluded"]
copied, errors = 0, []
for row in to_copy.itertuples(index=False):
    dst = os.path.join(DATASET_OUT_DIR, row.split, row.class_name, f"{row.source}_{row.filename}")
    try:
        shutil.copy2(row.filepath, dst)
        copied += 1
    except Exception as e:
        errors.append((row.filepath, str(e)))

print(f"Copied {copied}/{len(to_copy)} files in {time.time()-t0:.1f}s")
if errors:
    print(f"WARNING: {len(errors)} copy errors, e.g. {errors[:5]}")
else:
    print("No copy errors.")

excluded_n = int((manifest_df["split"] == "excluded").sum())
print(f"Excluded (not materialized, by design): {excluded_n}")

## 17. Verification that every (non-excluded) manifest entry points to an existing file

In [ ]:
usable_df = manifest_df[manifest_df["split"] != "excluded"]
missing_src, missing_dst = [], []
for row in usable_df.itertuples(index=False):
    if not os.path.isfile(row.filepath):
        missing_src.append(row.filepath)
    dst = os.path.join(DATASET_OUT_DIR, row.split, row.class_name, f"{row.source}_{row.filename}")
    if not os.path.isfile(dst):
        missing_dst.append(dst)

print(f"Usable manifest rows: {len(usable_df)}")
print(f"Missing source files: {len(missing_src)}")
print(f"Missing materialized (train/val/test) files: {len(missing_dst)}")
if not missing_src and not missing_dst:
    print("PASS: every usable manifest entry has both a source file and a materialized copy on disk.")
else:
    print("FAIL: see missing_src / missing_dst for details.")
    print(missing_src[:5], missing_dst[:5])

## 18. Save the final `split_manifest.csv`

Contains only the usable (non-excluded) images with their assigned split --
the two label-conflict-excluded images are intentionally not in this file;
they are recorded separately in `master_manifest.csv` with their exclusion
reason (section 19).

In [ ]:
split_manifest_path = os.path.join(RESULTS_OUT_DIR, "split_manifest.csv")
usable_df = manifest_df[manifest_df["split"] != "excluded"]
usable_df.to_csv(split_manifest_path, index=False)
print("Saved:", split_manifest_path)
print("Rows :", len(usable_df),
      f"(+ {int((manifest_df['split']=='excluded').sum())} excluded -- see master_manifest.csv)")

## 19. Save other verification CSVs/results

`master_manifest.csv` now includes **every** discovered image (2,470 rows),
including the 2 excluded ones -- with `excluded`, `exclusion_reason`,
`group_id`, and `split` (`"excluded"` for those 2 rows) columns, so the
exclusion decision is fully traceable rather than the images simply
disappearing from the record.

In [ ]:
paths_written = {}

p = os.path.join(RESULTS_OUT_DIR, "master_manifest.csv")
manifest_df.to_csv(p, index=False)
paths_written["master_manifest"] = p

p = os.path.join(RESULTS_OUT_DIR, "near_duplicate_candidates.csv")
near_dup_df.to_csv(p, index=False)
paths_written["near_duplicate_candidates"] = p

p = os.path.join(RESULTS_OUT_DIR, "exclusion_log.csv")
exclusion_df.to_csv(p, index=False)
paths_written["exclusion_log"] = p

p = os.path.join(RESULTS_OUT_DIR, "expected_vs_actual.json")
with open(p, "w") as f:
    json.dump({"expected_reference": EXPECTED_REFERENCE,
               "actual": {"total": TOTAL, "normal": NORMAL, "glaucoma": GLAUCOMA},
               "match": all_pass}, f, indent=2)
paths_written["expected_vs_actual"] = p

for name, p in paths_written.items():
    print(f"Saved {name:28s} -> {p}")

## 20. Final verification summary

In [ ]:
usable_df = manifest_df[manifest_df["split"] != "excluded"]
n_excluded = int((manifest_df["split"] == "excluded").sum())
group_split_counts = usable_df.groupby("group_id")["split"].nunique()
crossing = int((group_split_counts > 1).sum())

print("=" * 60)
print("NOTEBOOK 1 -- FINAL VERIFICATION SUMMARY")
print("=" * 60)
print(f"Sources used              : {list(usable_sources.keys())}")
print(f"Total labeled images (raw): {TOTAL}")
print(f"  Normal                  : {NORMAL} ({NORMAL/TOTAL:.1%})")
print(f"  Glaucoma                : {GLAUCOMA} ({GLAUCOMA/TOTAL:.1%})")
print(f"Matches prior reference check (raw counts): {all_pass}")
print(f"Exact (MD5) duplicates found   : {len(exact_dups)}")
print(f"Confirmed near-duplicates      : {len(near_dup_df)}")
print(f"Excluded (conflicting-label near-duplicates): {n_excluded}")
print(f"Usable images after exclusion  : {len(usable_df)}")
print(f"  Normal                       : {(usable_df['label']==0).sum()} ({(usable_df['label']==0).mean():.1%})")
print(f"  Glaucoma                     : {(usable_df['label']==1).sum()} ({(usable_df['label']==1).mean():.1%})")
print(f"Train / Val / Test sizes       : "
      f"{(usable_df['split']=='train').sum()} / "
      f"{(usable_df['split']=='val').sum()} / "
      f"{(usable_df['split']=='test').sum()}")
print(f"Train / Val / Test proportions : "
      f"{(usable_df['split']=='train').mean():.1%} / "
      f"{(usable_df['split']=='val').mean():.1%} / "
      f"{(usable_df['split']=='test').mean():.1%}")
print(f"Same-eye/near-duplicate groups crossing a split boundary: {crossing} (must be 0)")
print(f"Missing files (source/materialized): {len(missing_src)} / {len(missing_dst)}")
print(f"Split manifest saved at: {split_manifest_path}")
print(f"Materialized dataset at: {DATASET_OUT_DIR}")
print("=" * 60)